# README
- **Author**: RUEI-HSIANG, CHANG（B12303030；AI 協助資料蒐集整理）
- **Created At**: 2026-10-17
- **Last Modified At**: 2026-10-17

---
## What does this file do?
遠見媒體彙整檔逐一核對81人姓名、縣市、登記推薦政黨（明列縮寫與温／溫異體字），吻合不代表逐項學位已查證。空陣列與「尚未公布」仍為缺值。遠見原文學歷保存於observations之education_gvm_compilation；僅在原CNA學歷空白時補入education_media_supplement並明標學位待核實。新增羅佩秦學歷說明，原視報導可支持陸官正期53期畢業證書的說法，但未獨立確認其兩項學士學位；不改education_2026或教育分類。出生年無新增，仍缺羅佩秦、魏造文。GVM政見為媒體編輯摘要，不合併到候選人官網原文段落。
本次再查原缺性別的25人，新增19筆可追溯觀察（政府／議會／政黨／專業學會8人、媒體11人），主表80/81（新增5人依使用者提供，未獨立查證）；其中葉人文依媒體「大港男兒」明示描述，並非戶籍性別欄。原缺6人中，使用者補充王肇民、洪方隆、朱定瑀、羅佩秦、李吳穎智為男，以user_provided標記；魏造文回覆未知，主欄仍保留空白，sex_evidence_status記錄user_reported_unknown。社群目錄暫存值保留原來源。李文良僅取得官方名冊搜尋索引片段，林國漳採web工具抽出PDF文字（直接下載403），限制分列於sex_evidence_status。性別欄為公開來源標記，不代表已查驗2026戶籍資料。逐人來源與未解缺口見data/raw/sex_verification/sex_search_audit_25.csv。

補充 81 位 2026 縣市長候選人的性別、出生日期、年度年齡、學歷來源文字、現任資料、兩種首次參選證據、競選網站與社群連結。只蒐集與整理，不估計統計模型、不決定尚未確認的分群或樣本規則。

原登記名冊與舊版整理文件保留為第一版，不覆寫。姓名相同不足以辨識人物：歷史 CEC 紀錄須與官方公報或媒體資料的完整生日一致才採納；媒體交叉核對依據明列 source tier。李四川的同名里長、許淑華的同名議員不得誤併。未命中不填終生首次=1；確實找到過去參選，才填首次=0。

---
## What does this file take?
固定版本登記名冊、146 份官方 CEC 全公職歷史候選人原始 CSV、2024 不分區公報、19 位立委官方頁面、政府／政黨簡介及候選人自設網站、中央社 79 人背景字典、81 人逐人查詢紀錄。`data/raw/enrichment_input_manifest.json` 逐檔保存路徑與固定 SHA-256，下方另外核對 manifest 自身及處理程式的 hash。失敗下載保留錯誤，不當成成功抓取。原始資料文件：[sources_2026_documentation.ipynb](../raw/sources_2026_documentation.ipynb)。

---
## What does this file output?
在 `data/processed/` 產生 candidates_2026_enriched、candidate_profile_observations、candidate_all_office_history、candidate_campaign_sources、candidate_campaign_text_blocks、candidate_collection_coverage、candidate_source_search_log、candidate_source_issues 八份 CSV。各檔為 preparation/evidence，非已完成策略分析。schema、列數與 SHA-256 於最後一格依實際儲存重讀檢查列印；文件：[candidate_enrichment_documentation.ipynb](../processed/candidate_enrichment_documentation.ipynb)。後續分析須使用通過核對的正式版本。

中央社資料只供補充與核對，不取代官方推薦政黨或自動認定目前職務。學歷保留 current profile／historical CEC／media supplement 三個來源欄位，不能把較早資料當成 2026 最終學歷。官方頁面也可能未更新、學位可能存在爭議，不在本次自動認證。社群網址連結不表示已取得完整貼文；官網段落不等同獨立政見項目。


In [1]:
from pathlib import Path
import hashlib
import pandas as pd


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "AGENTS.md").is_file() and (candidate / "templates").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook within the student project containing AGENTS.md and templates/.")


PROJECT_ROOT = find_project_root()


def project_file(relative_path, area, must_exist=True):
    # Explicit project-relative paths only; never select the first matching file.
    rel = Path(relative_path)
    if rel.is_absolute() or ".." in rel.parts or "<" in str(rel):
        raise ValueError(f"Replace the placeholder with an exact project-relative path: {relative_path}")
    path = (PROJECT_ROOT / rel).resolve()
    allowed = (PROJECT_ROOT / "data" / area).resolve()
    if allowed == PROJECT_ROOT or not allowed.is_relative_to(PROJECT_ROOT):
        raise ValueError(f"data/{area}/ resolves outside the project root.")
    if not path.is_relative_to(allowed):
        raise ValueError(f"Expected a file inside data/{area}/: {relative_path}")
    if must_exist and not path.is_file():
        raise FileNotFoundError(path)
    return path


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def verify_input_sha256(path, expected):
    # Bind this run to the exact file inspected when the notebook was prepared.
    if (not isinstance(expected, str) or len(expected) != 64
            or any(char not in "0123456789abcdefABCDEF" for char in expected)):
        raise ValueError("Fill the expected SHA-256 from the inspected input before loading data.")
    actual = sha256_file(path)
    if actual != expected.lower():
        raise ValueError(f"Input changed: identify the new version before loading {path.name}.")
    return actual

import json, importlib.util
MANIFEST_HASH='ef903cf53e78ce5615b922bdf42193b8a57d92dd84f45b4ee6d4b5127109709c'
verify_input_sha256(project_file("data/raw/enrichment_input_manifest.json","raw"),MANIFEST_HASH)
INPUT_HASHES=json.loads(project_file("data/raw/enrichment_input_manifest.json","raw").read_text())
CODE_PATH=PROJECT_ROOT/"data/code/enrich_candidate_sources.py"
verify_input_sha256(CODE_PATH,'a53a9527334a22612926d2437fc68c1eaa853edf2c3e9178401481e6ddc3b99c')
spec=importlib.util.spec_from_file_location("enrich_candidate_sources",CODE_PATH)
module=importlib.util.module_from_spec(spec); spec.loader.exec_module(module)

In [2]:
outputs,verification=module.build(INPUT_HASHES)
for key,result in verification.items():print(key,result['rows'],result['sha256'])


candidates_2026_enriched 81 291a31d7588befded91e8dce0caf7a64cc4cecc927bb4c5fbecd218f6c7281b8
candidate_profile_observations 593 fc5ef77666a1cf772203269222957e28c798fc5000f0b81ba68e76d78f8c6037
candidate_all_office_history 247 6766413aa9a810f8109adba0a1b061432f193fdfbb5fc312563bab02dce5ce70
candidate_campaign_sources 96 682e1fffd9671ae2182548f07a0894b489d8edd1803f4b723106b991115341bd
candidate_campaign_text_blocks 459 3b97686ff977c6f58af3ad84bbd305559d680dcb99fa62a9bb49721c8fa0bc42
candidate_collection_coverage 81 3ec1afc101e6cb169f9bd56a7784336d1a7da34b09b40a3864a7f30b8cd45977
candidate_source_search_log 81 d117281cc54916fe67b8363c8ed0e50915cb33f5e77a0d24aa3477eb3bb60682
candidate_source_issues 8 830defffbfa919386dc1b9bfe67aba7bdf2843442b0958adbb319509963b0d9d


In [3]:
coverage=outputs['candidate_collection_coverage']
display(coverage.select_dtypes('number').sum().to_frame('candidates_with_field_or_count'))
display(outputs['candidates_2026_enriched'][['county_city','candidate_name','sex_2026','birth_date_verified','education_profile_as_captured','first_time_mayor_lifetime','first_time_any_elected_office']].head(10))
display(outputs['candidate_all_office_history'].identity_match_status.value_counts())


,candidates_with_field_or_count
sex_available,80
primary_birth_date_available,55
media_birth_year_available,79
primary_profile_education_available,25
historical_cec_education_available,50
media_education_available,78
first_time_mayor_resolved,21
first_time_any_office_resolved,56
official_facebook_link_found,8
campaign_website_found,8


,county_city,candidate_name,sex_2026,birth_date_verified,education_profile_as_captured,first_time_mayor_lifetime,first_time_any_elected_office
0,臺北市,郭璽,男,1958-08-31,,,0
1,臺北市,沈伯洋,男,1982-06-07,加州大學爾灣分校犯罪與法律社會學博士；賓州大學法學碩士；台大法律法學碩士,,0
2,臺北市,蕭文乾,男,1974-09-27,,,0
3,臺北市,唐新民,男,1950-12-25,,0,0
4,臺北市,蔣萬安,男,1978-12-26,美國賓夕法尼亞大學法學院法律博士；國立政治大學法律系、外交系雙學士,0,0
5,臺北市,林志成,男,1965-05-28,,,0
6,新北市,蘇巧慧,女,1976-04-05,美國賓州大學法律碩士；美國波士頓大學法律碩士；台大法律系司法組；北一女中,,0
7,新北市,李四川,男,,國立臺北科技大學電機研究所碩士,,
8,新北市,蘇輝湟,男,1976-11-10,,,0
9,桃園市,張善政,男,1954-06-24,,0,0


identity_match_status
dob_corroborated_by_media       132
name_only_pending_identity       66
rejected_birth_date_mismatch     49
Name: count, dtype: int64

In [4]:
for key,result in verification.items():
    p=project_file(result['path'],'processed'); df=pd.read_csv(p,dtype=str,keep_default_na=False)
    assert len(df)==result['rows'] and list(df.columns)==result['columns']
    assert sha256_file(p)==result['sha256']
print('All serialized outputs re-read and verified. Social post totals remain uncollected, not zero.')
(PROJECT_ROOT/'data/code/enrichment_verification_2026.json').write_text(json.dumps(verification,ensure_ascii=False,indent=2))


All serialized outputs re-read and verified. Social post totals remain uncollected, not zero.


5404